## IDHM por UF (2022 a 2024)

**Origem:** `idhm/adh_radar_base_2012_2024.xlsx`, aba `Total`. Essa aba tem uma linha por território e ano. Usamos só as linhas de `AGREGACAO == 'Unidade da Federação'`, que correspondem às 27 UFs.

**Resultado:** uma linha por ano e UF, com o IDHM, suas dimensões e componentes (escolaridade, frequência, analfabetismo, atraso escolar, renda, GINI e THEIL). O código é o código IBGE da UF, em `state_code`. As colunas estão em inglês e minúsculo.

**Atenção:**
- O arquivo do Radar termina em **2024**. O recorte é de 2022 a 2024.
- Na renda por quintil e décimo, a fonte traz os quintis 1 a 5 e o décimo mais rico (`income_per_capita_decile_10`). Os décimos 6 a 9 não existem na fonte.

Gerado por Claude.

In [1]:
import pandas as pd
import re

In [2]:
# Lê a aba Total do Radar, que tem uma linha por território e ano
adh = pd.read_excel("idhm/adh_radar_base_2012_2024.xlsx", sheet_name="Total")

# Mantém só as Unidades da Federação e os anos de 2022 a 2024
adh_anos = adh[(adh['AGREGACAO'] == 'Unidade da Federação') & (adh['ANO'].between(2022, 2024))]

adh_anos.head(10)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
# Padrão das colunas pedidas: IDHM e dimensões, componentes, grupos T_*, ANOSEST, RDPC, GINI e THEIL
regex = r'^(IDHM|IDHM_E|IDHM_L|IDHM_R|I_ESCOLARIDADE|I_FREQ_PROP|T_FREQ.*|T_FUND.*|T_MED.*|T_ANALF.*|T_ATRASO_.*|ANOSEST|RDPC|RDPCT|RDPC\d+|GINI|THEIL)$'

# Seleciona as colunas da aba que casam com o padrão
colunas = [c for c in adh.columns if re.match(regex, str(c))]

# Mostra quantas colunas foram encontradas
print(colunas)

['IDHM', 'IDHM_L', 'IDHM_E', 'IDHM_R', 'I_ESCOLARIDADE', 'I_FREQ_PROP', 'T_FREQ5A6', 'T_FUND11A13', 'T_FUND15A17', 'T_MED18A20', 'T_FUND18M', 'ANOSEST', 'T_ANALF15M', 'T_ANALF18M', 'T_ANALF25M', 'T_FREQ6A14', 'T_FREQ15A17', 'T_FREQ18A24', 'T_FUND18A24', 'T_FUND25M', 'T_MED25M', 'T_ATRASO_2_BASICO', 'T_ATRASO_2_FUND', 'RDPC', 'RDPCT', 'RDPC1', 'RDPC2', 'RDPC3', 'RDPC4', 'RDPC5', 'RDPC10', 'GINI', 'THEIL']


In [ ]:
# Dicionário de renomeação: nomes originais do Radar para inglês em minúsculo, no padrão snake_case
renomear = {
    # Identificação da UF e do ano
    'ANO': 'year',
    'CODIGO': 'state_code',
    'NOME': 'state_name',

    # IDHM geral e suas dimensões
    'IDHM': 'hdi',
    'IDHM_L': 'hdi_life_expectancy',
    'IDHM_E': 'hdi_education',
    'IDHM_R': 'hdi_income',

    # Subíndices de educação
    'I_ESCOLARIDADE': 'education_subindex',
    'I_FREQ_PROP': 'school_attendance_subindex',

    # Taxas de frequência escolar por faixa etária
    'T_FREQ5A6': 'school_attendance_rate_5_6',
    'T_FREQ6A14': 'school_attendance_rate_6_14',
    'T_FREQ15A17': 'school_attendance_rate_15_17',
    'T_FREQ18A24': 'school_attendance_rate_18_24',

    # Taxas de conclusão do fundamental
    'T_FUND11A13': 'elementary_completion_rate_11_13',
    'T_FUND15A17': 'elementary_completion_rate_15_17',
    'T_FUND18A24': 'elementary_completion_rate_18_24',
    'T_FUND18M': 'elementary_completion_rate_18_plus',
    'T_FUND25M': 'elementary_completion_rate_25_plus',

    # Taxas de conclusão do médio
    'T_MED18A20': 'high_school_completion_rate_18_20',
    'T_MED25M': 'high_school_completion_rate_25_plus',

    # Taxas de analfabetismo
    'T_ANALF15M': 'illiteracy_rate_15_plus',
    'T_ANALF18M': 'illiteracy_rate_18_plus',
    'T_ANALF25M': 'illiteracy_rate_25_plus',

    # Atraso idade-série de 2 anos ou mais
    'T_ATRASO_2_BASICO': 'age_lag_2y_basic_rate',
    'T_ATRASO_2_FUND': 'age_lag_2y_elementary_rate',

    # Anos de estudo esperados aos 18 anos
    'ANOSEST': 'expected_years_of_schooling',

    # Renda per capita: média, média sem renda nula, quintis e décimo mais rico
    'RDPC': 'income_per_capita_mean',
    'RDPCT': 'income_per_capita_nonzero_mean',
    'RDPC1': 'income_per_capita_quintile_1',
    'RDPC2': 'income_per_capita_quintile_2',
    'RDPC3': 'income_per_capita_quintile_3',
    'RDPC4': 'income_per_capita_quintile_4',
    'RDPC5': 'income_per_capita_quintile_5',
    'RDPC10': 'income_per_capita_decile_10',

    # Desigualdade de renda
    'GINI': 'gini_index',
    'THEIL': 'theil_index',
}

# Monta o dataset com uma linha por ano e UF, convertendo o código da UF para inteiro
idhm_uf = adh_anos[['ANO', 'CODIGO', 'NOME'] + colunas].assign(CODIGO=lambda df: df['CODIGO'].astype(int))

# Ordena por ano e código da UF, reinicia o índice e aplica os nomes em inglês
idhm_uf = idhm_uf.sort_values(['ANO', 'CODIGO']).reset_index(drop=True).rename(columns=renomear)

# Mostra os anos presentes e o tamanho final do dataset
print('Anos:', sorted(idhm_uf['year'].unique().tolist()), '| shape:', idhm_uf.shape)
idhm_uf.head()

Anos: [2022, 2023, 2024] | shape: (81, 36)


,year,state_code,state_name,hdi,hdi_life_expectancy,hdi_education,hdi_income,education_subindex,school_attendance_subindex,school_attendance_rate_5_6,...,income_per_capita_mean,income_per_capita_nonzero_mean,income_per_capita_quintile_1,income_per_capita_quintile_2,income_per_capita_quintile_3,income_per_capita_quintile_4,income_per_capita_quintile_5,income_per_capita_decile_10,gini_index,theil_index
0,2022,11,Rondônia,0.762,0.832,0.743,0.715,0.633,0.805,90.56,...,682.74,695.89,151.83,326.51,495.62,711.24,1733.47,2393.71,0.455,0.351
1,2022,12,Acre,0.732,0.837,0.695,0.674,0.653,0.717,84.32,...,530.92,543.87,74.44,188.34,325.59,541.75,1526.86,2169.54,0.536,0.507
2,2022,13,Amazonas,0.757,0.830,0.778,0.671,0.733,0.801,90.35,...,520.67,525.39,95.57,209.76,331.65,523.47,1445.29,2065.07,0.505,0.446
3,2022,14,Roraima,0.763,0.805,0.813,0.680,0.779,0.830,92.50,...,549.77,560.27,100.01,226.27,363.37,579.26,1483.80,2075.96,0.494,0.422
4,2022,15,Pará,0.734,0.837,0.701,0.674,0.628,0.741,96.11,...,531.15,537.02,96.17,217.07,335.65,515.36,1492.35,2165.76,0.514,0.465


In [ ]:
# Salva o dataset final em CSV, no mesmo padrão separado por ponto e vírgula dos outros arquivos
idhm_uf.to_csv("idhm/idhm_uf_2022_2024.csv", sep=";", index=False, encoding="utf-8")